# Figure S2 — mixed hiPSC–K562 Parse v3 compatibility panels

This notebook reproduces **Figure S2 panels i, j, l, m, and n only**. These are the panels derived from the mixed hiPSC–K562 Parse Evercode v3 experiment.

It deliberately excludes:

- panel a (workflow schematic),
- panels b–f (earlier hiPSC circular-capture versus CROP-seq benchmark),
- panels g–h (TeloHAEC benchmark), and
- panel k (gene-detection saturation from the earlier hiPSC-only benchmark).

The computational source is `recovery/raw_downloads/250904.Parse.v3.iPSC.K562.copy.ipynb`. The producer reads the original Split-pipe expression matrix, long guide-UMI table, filtered guide reference, and four transcript saturation reports from `250730_eTN10_Parsev3_mixediPSC_K562`.


In [ ]:
# Repository-path bootstrap for local support modules.
from pathlib import Path
import sys


def _find_repo_root(start=None):
    start = Path(start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "src" / "cc_perturbseq").is_dir():
            return candidate
    raise FileNotFoundError(
        "Could not locate the repository root containing src/cc_perturbseq. "
        "Run this notebook from within the repository checkout."
    )


REPO_ROOT = _find_repo_root()
for _path in (REPO_ROOT, REPO_ROOT / "src"):
    if str(_path) not in sys.path:
        sys.path.insert(0, str(_path))
print("Repository:", REPO_ROOT)


In [ ]:
import json
import os
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import yaml
from IPython.display import display

from src.cc_perturbseq.figure_notebook import (
    initialize_figure,
    panel_directory,
    sha256_file,
    write_receipt,
)
from src.cc_perturbseq.figure_s02_parse_mix import (
    CAPTURE_PRIMER_ORDER,
    CELL_TYPE_ORDER,
    MANUSCRIPT_TRANSCRIPT_ORDER,
    PRIMER_ORDER,
    SOURCE_NOTEBOOK,
    plot_panel_i,
    plot_panel_j,
    plot_panel_l,
    plot_panel_m,
    plot_panel_n,
    prepare_parse_mix_benchmark,
)

FIGURE = {
    "figure_id": "Figure_S02",
    "title": "Figure S2 — Development and benchmarking of CC-Perturb-seq",
    "scope": "Panels i, j, l, m and n only — mixed hiPSC–K562 Parse v3 experiment",
    "manuscript": "Working CC-Perturb-seq EC Differentiation - Text V3",
    "panels": [
        {
            "panel": "i",
            "description": "Matched-pool guide recovery by transcriptomic cell type and primer",
            "type": "code",
            "required_inputs": [],
            "candidate_sources": [SOURCE_NOTEBOOK],
        },
        {
            "panel": "j",
            "description": "Guide-pool specificity by transcriptomic cell type and primer",
            "type": "code",
            "required_inputs": [],
            "candidate_sources": [SOURCE_NOTEBOOK],
        },
        {
            "panel": "l",
            "description": "Top-guide UMI capture including the no-guide-primer control",
            "type": "code",
            "required_inputs": [],
            "candidate_sources": [SOURCE_NOTEBOOK],
        },
        {
            "panel": "m",
            "description": "Transcriptome UMI recovery and read-depth saturation",
            "type": "code",
            "required_inputs": [],
            "candidate_sources": [SOURCE_NOTEBOOK],
        },
        {
            "panel": "n",
            "description": "hiPSC-pool versus K562-pool guide mixing after the 3-UMI cell-guide filter",
            "type": "code",
            "required_inputs": [],
            "candidate_sources": [SOURCE_NOTEBOOK],
        },
    ],
}
ctx = initialize_figure(FIGURE)

scope_config_path = REPO_ROOT / "config" / "figure_s02_parse_mix.yaml"
scope_config = yaml.safe_load(scope_config_path.read_text(encoding="utf-8"))
EXPERIMENT_ROOT = Path(
    os.environ.get(
        scope_config["override_environment_variable"],
        scope_config["experiment_root"],
    )
).expanduser()
SOURCE_NOTEBOOK_PATH = REPO_ROOT / scope_config["source_notebook"]


def export_matplotlib(figure, panel, stem):
    out = panel_directory(ctx, panel)
    result = {}
    for suffix in ("pdf", "svg", "png"):
        path = out / f"{stem}.{suffix}"
        kwargs = {"dpi": 300} if suffix == "png" else {}
        figure.savefig(
            path,
            bbox_inches="tight",
            facecolor="white",
            transparent=False,
            **kwargs,
        )
        result[suffix] = str(path)
    return result


display(
    {
        "scope": FIGURE["scope"],
        "experiment_root": str(EXPERIMENT_ROOT),
        "source_notebook": str(SOURCE_NOTEBOOK_PATH),
        "source_notebook_sha256": sha256_file(SOURCE_NOTEBOOK_PATH),
        "configuration": str(scope_config_path),
        "environment_override": scope_config["override_environment_variable"],
    }
)
ctx["panels"]


## Reconstruct the mixed-cell benchmark metrics

The producer applies the source notebook's cell-QC cutoffs, repeats its normalized-expression/HVG/PCA/neighborhood/Leiden workflow, assigns the two transcriptomic lineages using the marker sets specified in the manuscript, and streams the long guide-UMI table without constructing a dense cell-by-113,275-guide matrix.

Historical panel medians are exported as **diagnostics only**; small differences do not stop execution.


In [ ]:
result = prepare_parse_mix_benchmark(EXPERIMENT_ROOT)

provenance_path = ctx["output_dir"] / "Figure_S02_parse_mix_provenance.json"
provenance_path.write_text(
    json.dumps(result.provenance, indent=2, sort_keys=True) + "\n",
    encoding="utf-8",
)
cluster_path = ctx["output_dir"] / "Figure_S02_transcriptomic_cluster_annotation.csv"
result.cluster_annotation.to_csv(cluster_path, index=False)
diagnostic_path = ctx["output_dir"] / "Figure_S02_manuscript_median_diagnostics.csv"
result.median_diagnostics.to_csv(diagnostic_path, index=False)

display(pd.Series(result.provenance, dtype="object"))
display(result.cluster_annotation)
display(result.median_diagnostics)


## Panel i — matched-pool guide recovery

For transcriptome-assigned K562 and hiPSC cells, guide UMIs are summed from the guide pool matching the assigned lineage. CS1, DC1, and Dual primer conditions are shown separately.


In [ ]:
fig_i, source_i = plot_panel_i(result.cell_metrics)
out_i = panel_directory(ctx, "i")
outputs_i = export_matplotlib(fig_i, "i", "Figure_S02i_matched_pool_guide_recovery")
source_i_path = out_i / "Figure_S02i_matched_pool_guide_recovery_source.csv.gz"
source_i.loc[:, [
    "sample", "cell_type", "matched_pool_guide_umis",
    "K562_guide_umis", "hiPSC_guide_umis",
]].to_csv(source_i_path, compression="gzip")
diagnostic_i_path = out_i / "Figure_S02i_manuscript_median_diagnostics.csv"
result.median_diagnostics.loc[
    result.median_diagnostics["metric"].eq("matched_pool_guide_umis")
].to_csv(diagnostic_i_path, index=False)
write_receipt(
    ctx,
    "i",
    status="SOURCE_AND_MANUSCRIPT_VERIFIED_IMPLEMENTATION",
    inputs={
        "experiment_root": str(EXPERIMENT_ROOT),
        "source_notebook": str(SOURCE_NOTEBOOK_PATH),
        "source_notebook_sha256": sha256_file(SOURCE_NOTEBOOK_PATH),
    },
    outputs={
        **outputs_i,
        "source_table": str(source_i_path),
        "median_diagnostics": str(diagnostic_i_path),
    },
    parameters={
        "cell_types": list(CELL_TYPE_ORDER),
        "primers": list(CAPTURE_PRIMER_ORDER),
        "metric": "sum of guide UMIs from the pool matching transcriptomic cell type",
        "historical_medians": "diagnostic only",
    },
)
display(fig_i)


## Panel j — guide-pool specificity

Specificity is the matched-pool guide-UMI sum divided by the combined hiPSC- and K562-pool guide-UMI sum for each transcriptome-assigned cell.


In [ ]:
fig_j, source_j = plot_panel_j(result.cell_metrics)
out_j = panel_directory(ctx, "j")
outputs_j = export_matplotlib(fig_j, "j", "Figure_S02j_guide_pool_specificity")
source_j_path = out_j / "Figure_S02j_guide_pool_specificity_source.csv.gz"
source_j.loc[:, [
    "sample", "cell_type", "guide_pool_specificity",
    "matched_pool_guide_umis", "K562_guide_umis", "hiPSC_guide_umis",
]].to_csv(source_j_path, compression="gzip")
diagnostic_j_path = out_j / "Figure_S02j_manuscript_median_diagnostics.csv"
result.median_diagnostics.loc[
    result.median_diagnostics["metric"].eq("guide_pool_specificity")
].to_csv(diagnostic_j_path, index=False)
write_receipt(
    ctx,
    "j",
    status="SOURCE_AND_MANUSCRIPT_VERIFIED_IMPLEMENTATION",
    inputs={"experiment_root": str(EXPERIMENT_ROOT)},
    outputs={
        **outputs_j,
        "source_table": str(source_j_path),
        "median_diagnostics": str(diagnostic_j_path),
    },
    parameters={
        "cell_types": list(CELL_TYPE_ORDER),
        "primers": list(CAPTURE_PRIMER_ORDER),
        "specificity": "matched-pool UMIs / (hiPSC-pool UMIs + K562-pool UMIs)",
        "zero_pool_total": "excluded as undefined",
        "historical_medians": "diagnostic only",
    },
)
display(fig_j)


## Panel l — top-guide capture

The most abundant guide UMI count per cell is shown for CS1, DC1, Dual, and the no-guide-primer background condition, separately for transcriptome-assigned K562 and hiPSC cells.


In [ ]:
fig_l, source_l = plot_panel_l(result.cell_metrics)
out_l = panel_directory(ctx, "l")
outputs_l = export_matplotlib(fig_l, "l", "Figure_S02l_top_guide_umi_capture")
source_l_path = out_l / "Figure_S02l_top_guide_umi_capture_source.csv.gz"
source_l.loc[:, [
    "sample", "cell_type", "top_guide_umis", "total_guide_umis",
]].to_csv(source_l_path, compression="gzip")
write_receipt(
    ctx,
    "l",
    status="SOURCE_AND_MANUSCRIPT_VERIFIED_IMPLEMENTATION",
    inputs={"experiment_root": str(EXPERIMENT_ROOT)},
    outputs={**outputs_l, "source_table": str(source_l_path)},
    parameters={
        "cell_types": list(CELL_TYPE_ORDER),
        "primers": list(PRIMER_ORDER),
        "metric": "maximum UMI count among valid guides in each cell",
        "no_guide_role": "background control",
    },
)
display(fig_l)


## Panel m — transcriptome recovery and saturation

The left subplot shows total transcriptome UMIs per QC-passing cell. The right subplot reads the four Split-pipe `tscp_counts_subsampled.csv` reports and plots median transcriptome UMIs per cell against mean assigned reads per cell.


In [ ]:
fig_m, transcript_m, saturation_m = plot_panel_m(
    result.cell_metrics,
    result.saturation,
)
out_m = panel_directory(ctx, "m")
outputs_m = export_matplotlib(fig_m, "m", "Figure_S02m_transcriptome_recovery_and_saturation")
transcript_m_path = out_m / "Figure_S02m_transcriptome_UMI_source.csv.gz"
saturation_m_path = out_m / "Figure_S02m_transcript_saturation_source.csv"
transcript_m.to_csv(transcript_m_path, compression="gzip")
saturation_m.to_csv(saturation_m_path, index=False)
write_receipt(
    ctx,
    "m",
    status="SOURCE_AND_MANUSCRIPT_VERIFIED_IMPLEMENTATION",
    inputs={"experiment_root": str(EXPERIMENT_ROOT)},
    outputs={
        **outputs_m,
        "transcript_source_table": str(transcript_m_path),
        "saturation_source_table": str(saturation_m_path),
    },
    parameters={
        "primer_order": list(MANUSCRIPT_TRANSCRIPT_ORDER),
        "left_metric": "total transcriptome UMIs per QC-passing cell",
        "right_x": "mean assigned reads per cell from Split-pipe report",
        "right_y": "median transcriptome UMIs per cell from Split-pipe report",
    },
)
display(fig_m)


## Panel n — guide-pool mixing

Individual cell-guide observations below 3 UMIs are set to zero before summing the hiPSC- and K562-pool guide counts per cell. The main panel is a two-dimensional logarithmic histogram with marginal cell-count histograms.


In [ ]:
fig_n, source_n = plot_panel_n(result.cell_metrics)
out_n = panel_directory(ctx, "n")
outputs_n = export_matplotlib(fig_n, "n", "Figure_S02n_guide_pool_mixing")
source_n_path = out_n / "Figure_S02n_guide_pool_mixing_source.csv.gz"
source_n.to_csv(source_n_path, compression="gzip")
write_receipt(
    ctx,
    "n",
    status="SOURCE_AND_MANUSCRIPT_VERIFIED_IMPLEMENTATION",
    inputs={"experiment_root": str(EXPERIMENT_ROOT)},
    outputs={**outputs_n, "source_table": str(source_n_path)},
    parameters={
        "cell_guide_minimum_umis": 3,
        "x": "sum of retained hiPSC-pool guide UMIs per cell + 1",
        "y": "sum of retained K562-pool guide UMIs per cell + 1",
        "axes": "logarithmic",
        "color": "log-normalized cell count per 2D bin",
    },
)
display(fig_n)


## Outputs

All vector/raster figures, source tables, and run receipts are written under `outputs.root/Figure_S02/panel_<letter>/`. The output root comes from the standard repository path configuration.


In [ ]:
print("Figure S2 mixed-cell outputs:", ctx["output_dir"])
for panel in ("i", "j", "l", "m", "n"):
    print(panel, panel_directory(ctx, panel))
